收集四元组步骤

根据你提供的文献（张俊义的《基于空间转录组数据的肿瘤代谢通讯生态位识别方法》），你要在 NicheCompass 模型（基于图神经网络的空间组学分析工具）的基础上进行二次开发，核心创新点是从传统的“配体-受体”通讯变更为**“肿瘤代谢通讯四元组”**的因果逻辑约束。

这个“四元组”的结构是：上游调控通路 (Upstream Pathway) → 关键代谢酶 (Key Enzyme) → 隐匿代谢物 (Metabolite) → 下游受体/转运蛋白 (Downstream Receptor)。

要在 NicheCompass 中跑通你的模型，获取和构建这个四元组是第一步，也是最核心的生物学特征工程。以下是获取并构建四元组的详细、实操性步骤：

第一阶段：确定核心代谢通讯类型（查阅文献）

由于代谢物（如乳酸、ATP等）在空间转录组中是测不到的，你必须依赖权威的生物学先验知识来建立这套规则。你首先需要圈定肿瘤微环境中几种最经典的代谢共生/通讯机制。

常见的四元组范例（供你直接参考或扩充）：

糖酵解/乳酸通讯（经典瓦伯格效应）：

通路：EGFR 或 HIF-1 信号通路

酶：LDHA, HK2

代谢物：乳酸 (Lactate)

受体：MCT1 (SLC16A1), MCT4 (SLC16A3)

腺苷通讯（免疫抑制微环境）：

通路：Hypoxia (缺氧通路)

酶：CD39 (ENTPD1), CD73 (NT5E)

代谢物：腺苷 (Adenosine)

受体：A2AR (ADORA2A), A2BR (ADORA2B)

谷氨酰胺通讯：

通路：c-Myc 通路

酶：GLS (谷氨酰胺酶)

代谢物：谷氨酸 / 谷氨酰胺

受体：SLC1A5 (ASCT2)

第二阶段：结构化获取四元组的四个组件（数据库挖掘）

你需要将上述概念转化为具体的基因列表，以便输入给计算机。

步骤 1：获取“隐匿代谢物 → 下游受体” (受体端 / Receiver)

这一步 NicheCompass 的原文其实已经用到了，你需要提取代谢物与受体的对应关系。

推荐数据库： MEBOCOST (NicheCompass原文明确提及使用的数据库)、CellChatDB (其 v2 版本包含了代谢通讯模块)、HMDB (Human Metabolome Database)。

操作： 从 MEBOCOST 下载其开源的代谢物-传感器（Metabolite-Sensor）互作矩阵。过滤出与肿瘤代谢高度相关的对子（如 Lactate-SLC16A1）。

步骤 2：获取“关键代谢酶 → 隐匿代谢物” (生产端 / Sender)

知道了受体和代谢物，需要找到是谁生产了这个代谢物。

推荐数据库： KEGG Pathway (Metabolism 模块)、Recon3D 或 Human Metabolic Atlas。

操作： 在 KEGG 中搜索该代谢物，找到将其作为“产物（Product）”的催化反应，记录下对应的酶（Enzyme）的基因名称（如 LDHA）。注意：只选择该代谢途径中的关键限速酶，不要选择泛表达的基础代谢酶。

步骤 3：获取“上游调控通路 → 关键代谢酶” (驱动端 / Sender)

根据你的设计，只有被致癌通路驱动的酶高表达，才算是真正的 Sender。

推荐数据库： PROGENy (文献中明确提到的通路评分算法库)、MSigDB (Hallmark gene sets)、KEGG (Signaling 模块)。

操作：

使用 PROGENy 提供的 14 条经典癌症信号通路（如 EGFR, Hypoxia, PI3K, TGFb 等）。

获取这 14 条通路的**靶基因集（Target genes）**或特征基因集。

第三阶段：将四元组整合为 NicheCompass 格式的代码实现

NicheCompass 原始模型接受的是 Source (Sender) 和 Target (Receiver) 的基因列表（用于构建 P_neighborhood 和 P_self 掩码矩阵）。你需要将四元组映射到这两个维度上。

映射逻辑：

Sender 端特征 (Source)： 包含【上游调控通路基因集】 + 【关键代谢酶基因】

Receiver 端特征 (Target)： 包含【下游受体基因】

具体操作步骤：

1. 建立一个 CSV/Excel 表格作为你的“先验知识库” (Prior Knowledge Base)
表头设计如下：

Program_Name: 通讯程序名称 (例如: EGFR_Lactate_Communication)

Pathway_Genes: PROGENy中的通路基因 (例如: EGFR, PIK3CA, AKT1, MTOR...)

Enzyme_Gene: 代谢酶基因 (例如: LDHA)

Metabolite: 代谢物 (仅作注释: Lactate)

Receptor_Gene: 受体基因 (例如: SLC16A1)

2. 编写 Python 脚本将其转换为 NicheCompass 支持的字典格式
NicheCompass 需要一个特定的字典结构来定义 Gene Programs (GPs)。

code
Python
download
content_copy
expand_less
import pandas as pd

# 假设你已经建好了上面的 CSV 文件
df_four_tuple = pd.read_csv("tumor_metabolic_four_tuple.csv")

custom_metabolic_programs = {}

for index, row in df_four_tuple.iterrows():
    program_name = row['Program_Name']
    
    # 获取 Sender 端的基因：通路基因 + 酶基因
    # (注意：按照文献思路，你其实在这一步不仅需要基因名，后续计算图权重时还需要它们的表达量乘积)
    pathway_genes = [g.strip() for g in row['Pathway_Genes'].split(',')]
    enzyme_gene = row['Enzyme_Gene'].strip()
    source_genes = pathway_genes + [enzyme_gene]
    
    # 获取 Receiver 端的基因：受体基因
    target_genes = [row['Receptor_Gene'].strip()]
    
    # 构建 NicheCompass 格式的 Program
    custom_metabolic_programs[program_name] = {
        "sources": source_genes,
        "targets": target_genes
    }

print("成功构建了", len(custom_metabolic_programs), "个四元组代谢通讯程序！")
第四阶段：给你的下一步修改建议（非常重要！）

如果你仔细阅读你提供的这篇论文，你会发现，仅仅修改“基因列表”是不够的。这篇论文对 NicheCompass 的核心算法架构做了修改：

原始 NicheCompass： Sender特征仅仅是 Source基因表达量的简单组合。

你的目标模型： 要求 通讯强度 = (Sender的通路活性 × 酶表达量) × (Receiver的受体表达量)。

你下一步需要做的修改规划：

数据预处理层修改： 你需要使用 Python 包 decoupleR 或 scanpy 结合 PROGENy，提前计算出每个细胞/Spot的通路活性得分（Pathway Score），并把这个得分作为一个**新的特征（Feature）**加入到 AnnData 对象中，而不仅仅是输入基因表达量。

图结构构建层修改（修改 NicheCompass 的 adjacency 计算逻辑）： 原始模型是通过物理距离和转录组相似性建图。你需要修改它的 DataLoader 或构图函数，利用你算出来的四元组分数（通路得分×酶表达量×受体表达量）去重置图的边权重（Edge Weights）。

总结：
你现在的首要任务是建表（Excel/CSV），把生物学上的 通路-酶-代谢物-受体 一一对应起来。有了这个四元组字典，你才能去修改 NicheCompass 里面处理 Prior Knowledge 的模块。

In [1]:
import pandas as pd
import re

# ==========================================
# 1. 肿瘤核心代谢物白名单 (基于 HMDB ID 匹配，避免名称不一致)
# ==========================================
# 这里精选了肿瘤微环境中最重要的几种代谢物
tumor_metabolites_target = {
    'HMDB0000190': 'L-Lactate',        # L-乳酸 (瓦伯格效应核心)
    'HMDB0001311': 'D-Lactate',        # D-乳酸
    'HMDB0000050': 'Adenosine',        # 腺苷 (免疫抑制)
    'HMDB0000538': 'ATP',              # ATP (危险信号/炎症)
    'HMDB0000641': 'L-Glutamine',      # 谷氨酰胺 (肿瘤氮源)
    'HMDB0000254': 'Succinate',        # 琥珀酸 (致癌代谢物/促炎)
    'HMDB0001220': 'Prostaglandin E2'  # PGE2 (免疫抑制/血管生成)
}

# ==========================================
# 2. 定义上游驱动通路 (PROGENy / Hallmark 经典通路)
# ==========================================
pathway_prior_knowledge = {
    'L-Lactate': {'pathway': 'Hypoxia/EGFR', 'genes':['HIF1A', 'EGFR', 'MYC']},
    'D-Lactate': {'pathway': 'Hypoxia/EGFR', 'genes': ['HIF1A', 'EGFR', 'MYC']},
    'Adenosine': {'pathway': 'Hypoxia', 'genes': ['HIF1A']},
    'ATP': {'pathway': 'Stress/Apoptosis', 'genes': ['BAX', 'CASP3']},
    'L-Glutamine': {'pathway': 'c-Myc', 'genes': ['MYC']},
    'Succinate': {'pathway': 'Hypoxia', 'genes': ['HIF1A']},
    'Prostaglandin E2': {'pathway': 'NFkB/Inflammation', 'genes':['NFKB1', 'RELA']}
}

# ==========================================
# 3. 读取数据并进行处理
# ==========================================
print("正在读取 MEBOCOST 数据库文件...")
df_sensor = pd.read_csv('/home/zhangjunyi/xiangmu/nichecompass-main/data/pre_data/human_met_sensor.tsv', sep='\t')
df_enzyme = pd.read_csv('/home/zhangjunyi/xiangmu/nichecompass-main/data/pre_data/metabolite_enzyme_reaction.tsv', sep='\t')

# 过滤出我们在白名单里的代谢物
df_sensor_filtered = df_sensor[df_sensor['HMDB_ID'].isin(tumor_metabolites_target.keys())]
df_enzyme_filtered = df_enzyme[df_enzyme['HMDB_ID'].isin(tumor_metabolites_target.keys())]

four_tuples_dict = {}  # 存放最终 NicheCompass 格式的字典
four_tuples_table =[] # 存放用于可视化检查的表格数据



正在读取 MEBOCOST 数据库文件...


In [2]:
print("开始构建代谢通讯四元组...")

# 遍历每一个目标代谢物
for hmdb_id, met_name in tumor_metabolites_target.items():
    
    # ------------------------------------------------
    # A. 提取 Receiver 端：受体 (Sensors)
    # ------------------------------------------------
    # 获取该代谢物对应的所有传感器基因
    sensors_raw = df_sensor_filtered[df_sensor_filtered['HMDB_ID'] == hmdb_id]['Gene_name'].dropna().unique()
    # 过滤掉可能的空值或无效值，如果需要，你也可以根据 Annotation 过滤只保留 Receptor/Transporter
    receptors =[s.strip() for s in sensors_raw if isinstance(s, str)]
    
    # ------------------------------------------------
    # B. 提取 Sender 端：关键合成酶 (Enzymes)
    # ------------------------------------------------
    # 必须满足 direction == 'product' (代表这个反应是生成该代谢物的)
    enzymes_raw_series = df_enzyme_filtered[
        (df_enzyme_filtered['HMDB_ID'] == hmdb_id) & 
        (df_enzyme_filtered['direction'] == 'product')
    ]['gene'].dropna()
    
    enzymes = set()
    for row in enzymes_raw_series:
        # 清洗类似 "LDHA[Unknown]; LDHB[Unknown]" 的数据
        genes = str(row).split(';')
        for g in genes:
            clean_gene = g.split('[')[0].strip() # 截取 '[' 前面的纯基因名
            if clean_gene:
                enzymes.add(clean_gene)
    enzymes = list(enzymes)
    
    # ------------------------------------------------
    # C. 组装：上游通路 + 酶 -> 代谢物 -> 受体
    # ------------------------------------------------
    if not receptors or not enzymes:
        # 如果受体或酶在数据库中缺失，跳过
        continue
        
    pathway_info = pathway_prior_knowledge.get(met_name, {'pathway': 'Unknown', 'genes': []})
    pathway_genes = pathway_info['genes']
    
    # 保存为 NicheCompass 所需的格式： 
    # Source = 通路基因 + 酶基因 ; Target = 受体基因
    program_name = f"{pathway_info['pathway']}_{met_name}_Communication"
    
    four_tuples_dict[program_name] = {
        "sources": list(set(pathway_genes + enzymes)),
        "targets": receptors
    }
    
    # 保存为展示表格
    four_tuples_table.append({
        'Program_Name': program_name,
        'Metabolite': met_name,
        'Upstream_Pathway': pathway_info['pathway'],
        'Pathway_Genes': ", ".join(pathway_genes),
        'Enzyme_Genes (Sender)': ", ".join(enzymes),
        'Receptor_Genes (Receiver)': ", ".join(receptors)
    })



开始构建代谢通讯四元组...


In [3]:
# ==========================================
# 4. 输出结果
# ==========================================
df_result = pd.DataFrame(four_tuples_table)
print("\n========== 成功提取的肿瘤代谢通讯四元组 ==========")
display(df_result) # 如果在普通python运行，请改为 print(df_result.to_markdown())

print("\n========== NicheCompass Program 示例 ==========")
# 打印其中一个 Program 的结构，这就是传入 NicheCompass 的格式
sample_program = list(four_tuples_dict.keys())[0]
print(f"Program: {sample_program}")
print(f"Sources (Sender): {four_tuples_dict[sample_program]['sources']}")
print(f"Targets (Receiver): {four_tuples_dict[sample_program]['targets']}")



========== 成功提取的肿瘤代谢通讯四元组 ==========


,Program_Name,Metabolite,Upstream_Pathway,Pathway_Genes,Enzyme_Genes (Sender),Receptor_Genes (Receiver)
0,Hypoxia/EGFR_D-Lactate_Communication,D-Lactate,Hypoxia/EGFR,"HIF1A, EGFR, MYC","HAGHL, HAGH","SLC16A1, HCAR1"
1,Hypoxia_Adenosine_Communication,Adenosine,Hypoxia,HIF1A,"NT5C1B, NT5C2, AHCYL1, AHCY, NT5C3, NT5E, NT5C...","ADORA1, ADORA2A, ADORA2B, ADORA3, SLC29A2, SLC..."
2,Stress/Apoptosis_ATP_Communication,ATP,Stress/Apoptosis,"BAX, CASP3","HIBCH, NME1, NME6, MAT1A, NME4, ACLY, NME7, NM...","P2RX1, P2RX2, P2RX3, P2RX4, P2RX5, P2RX6, P2RX..."
3,c-Myc_L-Glutamine_Communication,L-Glutamine,c-Myc,MYC,"PPAT, GLUL","SLC1A5, SLC38A2, SLC3A2, SLC7A5, SLC6A14, SLC6..."
4,Hypoxia_Succinate_Communication,Succinate,Hypoxia,HIF1A,"EGLN2, EGLN3, TET1, NO66, TET3, PLOD3, TMLHE, ...","SUCNR1, SLC13A5, TRPC6"
5,NFkB/Inflammation_Prostaglandin E2_Communication,Prostaglandin E2,NFkB/Inflammation,"NFKB1, RELA","PTGES2, PTGES, CBR3, PTGES3, CBR1","PTGER1, PTGER2, PTGER4, SLC22A1, SLC22A2, PTGER3"



========== NicheCompass Program 示例 ==========
Program: Hypoxia/EGFR_D-Lactate_Communication
Sources (Sender): ['MYC', 'EGFR', 'HAGHL', 'HIF1A', 'HAGH']
Targets (Receiver): ['SLC16A1', 'HCAR1']


In [4]:
import os
import json

# ==========================================
# 5. 将提取的四元组保存到指定位置
# ==========================================

# 设置你想保存的文件夹路径 (你可以修改为你电脑上的绝对路径，比如 'D:/My_Project/Data/')
# 这里默认保存在当前运行目录下的 'Metabolic_Programs' 文件夹中
output_dir = "/home/zhangjunyi/xiangmu/nichecompass-main/data/pre_data/siyuanzu"

# 如果文件夹不存在，则自动创建
# if not os.path.exists(output_dir):
#     os.makedirs(output_dir)

# ------------------------------------------------
# 输出格式 1：保存为 CSV 文件（用于人工精修和论文图表）
# ------------------------------------------------
csv_file_path = os.path.join(output_dir, "tumor_metabolic_four_tuples_raw.csv")
# 将 DataFrame 保存为 CSV，去除行索引，使用 utf-8-sig 编码防止 Excel 打开中文乱码
df_result.to_csv(csv_file_path, index=False, encoding='utf-8-sig')
print(f"✅ 表格已成功保存至: {csv_file_path} (请用 Excel 打开它进行人工精简)")

# ------------------------------------------------
# 输出格式 2：保存为 JSON 文件（用于直接输入 NicheCompass 代码）
# ------------------------------------------------
json_file_path = os.path.join(output_dir, "nichecompass_metabolic_programs_raw.json")
# 将 字典 保存为格式化的 JSON 文件
with open(json_file_path, 'w', encoding='utf-8') as f:
    json.dump(four_tuples_dict, f, indent=4, ensure_ascii=False)
print(f"✅ NicheCompass 字典已成功保存至: {json_file_path}")

✅ 表格已成功保存至: /home/zhangjunyi/xiangmu/nichecompass-main/data/pre_data/siyuanzu/tumor_metabolic_four_tuples_raw.csv (请用 Excel 打开它进行人工精简)
✅ NicheCompass 字典已成功保存至: /home/zhangjunyi/xiangmu/nichecompass-main/data/pre_data/siyuanzu/nichecompass_metabolic_programs_raw.json


这些表头是你刚刚通过 Python 脚本提取出来的**“肿瘤代谢通讯四元组”**的核心组件。

这 6 列数据不仅是一条完整的生物学故事线，更是将抽象的生物学概念转化为 NicheCompass 算法可读取的数学特征的桥梁。

我们可以按照“信息传递的先后顺序”来逐一拆解这些列的含义：

1. Program_Name (通讯程序名称)

字面意思：这个四元组的唯一名称/标签。

生物学意义：代表一种特定的代谢通讯机制。

在 NicheCompass 中的作用：NicheCompass 模型会把这个名字作为图神经网络（GNN）的一个潜在特征维度（Latent dimension）。模型训练完成后，你可以直接在空间切片上给这个 Program 打分，分数高的地方，就是发生了这种代谢通讯的“生态位”。

2. Metabolite (隐匿代谢物)

字面意思：在细胞间传递的小分子代谢物（如 Lactate乳酸、ATP、Adenosine腺苷等）。

生物学意义：这是通讯的**“信使”**。

在 NicheCompass 中的作用：纯注释作用。因为空间转录组（RNA-seq）测不到代谢物，所以这个词不会输入给模型。它是我们构建上下游基因的“逻辑锚点”。

3. Upstream_Pathway (上游调控通路)

字面意思：驱动该代谢物产生的致癌信号通路（如 Hypoxia 缺氧通路、EGFR 通路等）。

生物学意义：肿瘤细胞为什么会疯狂分泌乳酸？是因为它处于缺氧环境或发生了基因突变。加上这一层，是为了剔除假阳性——我们要找的是“被肿瘤通路异常激活的代谢”，而不是正常细胞的日常基础代谢。

在 NicheCompass 中的作用：同样是注释作用，帮助你归类和解释结果。

4. Pathway_Genes (通路基因)

字面意思：上述“上游调控通路”中核心的转录因子或关键驱动基因（如 HIF1A, MYC, EGFR）。

生物学意义：这些基因的高表达，代表了肿瘤细胞开启了“代谢重编程”的开关。

在 NicheCompass 中的作用：核心输入特征之一。它们将被划入通讯的 Sender 端（发送端）[1]。

5. Enzyme_Genes (Sender) (关键代谢酶 / 发送端)

字面意思：催化底物生成目标 Metabolite 的酶的基因名（如产生乳酸的 LDHA）。

生物学意义：这是通讯信使的**“生产车间”**。有了上游开关（Pathway Genes）的指令，车间（Enzyme）开始运作，疯狂生产代谢物并分泌到细胞外。

在 NicheCompass 中的作用：核心输入特征之一。
👉 在 NicheCompass 的代码字典中，Pathway_Genes 和 Enzyme_Genes 会被合并在一起，共同构成 sources（信号发送方）基因集[1][2]。只有当一个细胞同时高表达通路基因和酶基因时，模型才会认为它是一个合格的代谢源（Sender）。

6. Receptor_Genes (Receiver) (下游受体 / 接收端)

字面意思：细胞膜上用于结合或转运该代谢物的受体/转运蛋白的基因名（如乳酸转运蛋白 SLC16A1、腺苷受体 ADORA2A）。

生物学意义：这是通讯的**“接收天线”**。肿瘤微环境中的其他细胞（如巨噬细胞、T细胞或成纤维细胞）通过表达这些受体，感知到肿瘤分泌的代谢物，从而被“驯化”（例如 T细胞被乳酸抑制，巨噬细胞向M2型促癌方向极化）。

在 NicheCompass 中的作用：核心输入特征之一。
👉 在 NicheCompass 的代码字典中，这列基因构成 targets（信号接收方）基因集[1][2]。

💡 总结：它们是如何在算法里联动工作的？

以 “乳酸（L-Lactate）通讯” 为例，你表格里的这一行向 NicheCompass 讲述了这样一个故事：

“如果在空间切片上，细胞 A（肿瘤细胞）大量表达了 HIF1A 和 LDHA (Sources)，并且它旁边的细胞 B（免疫细胞）大量表达了 SLC16A1 (Targets)。那么由于它们物理位置挨得很近，算法就会判定：这里正在发生 Hypoxia_L-Lactate_Communication (Program_Name)。细胞A和细胞B共同构成了一个肿瘤代谢通讯生态位 (TMCN)。”

你提取出来的这张表，本质上就是为人工智能制定了这套**“基于因果逻辑的判断规则”**。

Sources
help
biorxiv.org
researchgate.net
Google Search Suggestions
Display of Search Suggestions is required when using Grounding with Google Search. Learn more
"NicheCompass" "Gene Programs" "Sources" "Targets"

In [10]:
#我们直接把你现在的四元组拆解、格式化，生成 NicheCompass 需要的两个最终文件。请直接运行这段代码，它会把逗号分隔的基因拆成单行，完美适配模型格式
import pandas as pd

# 1. 读取你已经提取好的 MEBOCOST 四元组数据
df = pd.read_csv('/home/zhangjunyi/xiangmu/nichecompass-main/data/pre_data/siyuanzu/MEBOCOST/tumor_metabolic_four_tuples_raw.csv')

# ==========================================
# 制作 1：tumor_metabolite_enzymes.tsv (发送端字典)
# ==========================================
# 提取代谢物和酶基因列
enzymes_df = df[['Metabolite', 'Enzyme_Genes (Sender)']].copy()

# MEBOCOST 里的基因是用逗号隔开的 (比如 "HAGHL, HAGH")
# 我们需要把它们拆分开，让每一个基因占单独的一行
enzymes_df['Enzyme_Genes (Sender)'] = enzymes_df['Enzyme_Genes (Sender)'].str.split(', ')
enzymes_df = enzymes_df.explode('Enzyme_Genes (Sender)')

# 重命名列名，严格按照 NicheCompass 要求的全小写
enzymes_df.columns = ['metabolite', 'enzyme']

# ==========================================
# 制作 2：tumor_metabolite_sensors.tsv (接收端字典)
# ==========================================
# 提取代谢物和受体列
sensors_df = df[['Metabolite', 'Receptor_Genes (Receiver)']].copy()

# 同样拆分受体基因 (比如 "SLC16A1, HCAR1")
sensors_df['Receptor_Genes (Receiver)'] = sensors_df['Receptor_Genes (Receiver)'].str.split(', ')
sensors_df = sensors_df.explode('Receptor_Genes (Receiver)')

# 重命名列名
sensors_df.columns = ['metabolite', 'sensor']

# ==========================================
# 清理并导出
# ==========================================
# 去除空值并去重
enzymes_df = enzymes_df.dropna().drop_duplicates()
sensors_df = sensors_df.dropna().drop_duplicates()

# 导出为 Tab 分隔的 TSV 文件
enzymes_df.to_csv('tumor_metabolite_enzymes.tsv', sep='\t', index=False)
sensors_df.to_csv('tumor_metabolite_sensors.tsv', sep='\t', index=False)

print(f"🎉 成功！你保留了全部 {len(df['Metabolite'].unique())} 种肿瘤关键代谢物。")
print("已在当前目录生成 'tumor_metabolite_enzymes.tsv' 和 'tumor_metabolite_sensors.tsv'。")

🎉 成功！你保留了全部 6 种肿瘤关键代谢物。
已在当前目录生成 'tumor_metabolite_enzymes.tsv' 和 'tumor_metabolite_sensors.tsv'。
